# Frame Embedding — SigLIP2 (Kaggle)

Notebook offline: only reads the attached Kaggle Dataset and writes `/kaggle/working`. It never connects to PostgreSQL, APIs, Cloudflare/R2, captioning, or ASR. Enable a Kaggle GPU and Internet (or attach the model cache) before the real run.


In [ ]:
# Configuration
from __future__ import annotations
import os
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/btc-keyframes")
OUTPUT_DIR = Path("/kaggle/working/frame_embedding_output")
KEYFRAME_FILE = INPUT_DIR / "keyframe.csv"  # accepts keyframes.csv if absent
FRAME_ROOT = INPUT_DIR / "keyframes"
VIDEO_START = 0; VIDEO_END = None  # half-open over sorted video_id
IMAGE_BATCH_SIZE = 256
MIN_BATCH_SIZE = 8
NUM_WORKERS = min(4, os.cpu_count() or 1)
ROWS_PER_SHARD = 25_000
INDEX_VERSION = 1
RUN_ID = None  # None => timestamp + config; never overwrite an existing run

MODEL_ID = "google/siglip2-base-patch16-224"
MODEL_REVISION = "a7d042728184c5fa87e2569ec1c4121cb48f9885"
MODEL_NAME = "siglip2-base-patch16-224"
MODEL_BACKEND = "transformers"
assert 0 <= VIDEO_START and (VIDEO_END is None or VIDEO_END >= VIDEO_START)
assert MIN_BATCH_SIZE > 0 and IMAGE_BATCH_SIZE >= MIN_BATCH_SIZE and ROWS_PER_SHARD > 0


In [ ]:
# Imports and runtime setup. Missing packages fail clearly; this notebook does not install packages.
import csv, hashlib, json, math, shutil, time, uuid
from datetime import datetime, timezone
from typing import Any

import faiss
import numpy as np
import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoProcessor

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. In Kaggle: Settings → Accelerator → GPU, then restart the session.")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda")
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
GPU_NAME = torch.cuda.get_device_name(0)

if not KEYFRAME_FILE.is_file():
    KEYFRAME_FILE = INPUT_DIR / "keyframes.csv"
if not KEYFRAME_FILE.is_file():
    raise FileNotFoundError(f"Neither keyframe.csv nor keyframes.csv exists under {INPUT_DIR}")

def atomic_json(path: Path, value: Any) -> None:
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")
    temp.replace(path)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def is_oom(exc: RuntimeError) -> bool:
    return "out of memory" in str(exc).lower()


In [ ]:
# Input validation and deterministic path resolution. Invalid input is represented only in failures.jsonl.
REQUIRED_COLUMNS = {"frame_id","video_id","shot_id","timestamp_ms","fps","frame_idx","source","n","pts_time","frame_path","width","height"}

def append_failure(path: Path, frame_id: Any, reason: str, image_path: Any) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps({"frame_id": str(frame_id or ""), "reason": reason, "path": str(image_path or "")}, ensure_ascii=False) + "\n")

def resolve_image(raw_path: str) -> Path | None:
    source = Path(raw_path)
    candidates = [source] if source.is_absolute() else []
    candidates += [FRAME_ROOT / source, FRAME_ROOT / source.name]
    return next((candidate for candidate in candidates if candidate.is_file()), None)

def load_rows(failures: Path) -> tuple[list[dict[str, Any]], int]:
    with KEYFRAME_FILE.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
        if missing: raise ValueError(f"keyframe CSV missing required columns: {sorted(missing)}")
        raw = list(reader)
    seen, valid, failure_count = set(), [], 0
    for line, row in enumerate(raw, 2):
        try:
            frame_id, video_id = row["frame_id"].strip(), row["video_id"].strip()
            if not frame_id or frame_id in seen: raise ValueError("frame_id must be non-empty and unique")
            if not video_id or len(video_id) > 15: raise ValueError("video_id must be non-empty and at most 15 chars")
            timestamp_ms, fps, frame_idx = int(row["timestamp_ms"]), float(row["fps"]), int(row["frame_idx"])
            if timestamp_ms < 0 or fps <= 0 or frame_idx < 0: raise ValueError("timestamp_ms/frame_idx must be non-negative and fps > 0")
            if row["source"].strip() not in {"official", "extracted"}: raise ValueError("source must be official or extracted")
            seen.add(frame_id)
            valid.append({**row, "frame_id": frame_id, "video_id": video_id, "shot_id": row["shot_id"].strip() or None, "timestamp_ms": timestamp_ms, "fps": fps, "frame_idx": frame_idx, "source": row["source"].strip(), "image_path": resolve_image(row["frame_path"])})
        except Exception as exc:
            append_failure(failures, row.get("frame_id"), f"invalid_row line {line}: {exc}", row.get("frame_path")); failure_count += 1
    videos = set(sorted({row["video_id"] for row in valid})[VIDEO_START:VIDEO_END])
    selected = [row for row in valid if row["video_id"] in videos]
    selected.sort(key=lambda row: (row["video_id"], row["frame_idx"], row["frame_id"]))
    return selected, failure_count


In [ ]:
# Streaming PIL Dataset and fixed-revision SigLIP2 image projection.
class FrameDataset(Dataset):
    def __init__(self, rows: list[dict[str, Any]]): self.rows = rows
    def __len__(self) -> int: return len(self.rows)
    def __getitem__(self, index: int):
        row = self.rows[index]
        if row["image_path"] is None: return index, None, "image_not_found"
        try:
            with Image.open(row["image_path"]) as image:
                return index, image.convert("RGB").copy(), None
        except (OSError, UnidentifiedImageError) as exc:
            return index, None, f"image_decode_error: {exc}"

def collate(samples):
    return samples

processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(DEVICE)
model.eval()

def encode_images(images: list[Image.Image], initial_size: int) -> tuple[np.ndarray, int]:
    vectors, cursor, size = [], 0, initial_size
    while cursor < len(images):
        batch = images[cursor:cursor + size]
        try:
            inputs = processor(images=batch, return_tensors="pt")
            inputs = {name: value.to(DEVICE, non_blocking=True) for name, value in inputs.items()}
            with torch.inference_mode(), torch.autocast("cuda", dtype=COMPUTE_DTYPE):
                output = model.get_image_features(**inputs)
            output = torch.nn.functional.normalize(output.float(), p=2, dim=1)
            value = np.ascontiguousarray(output.cpu().numpy(), dtype=np.float32)
            if not np.isfinite(value).all() or not np.allclose(np.linalg.norm(value, axis=1), 1.0, atol=1e-4):
                raise RuntimeError("model returned non-finite or non-normalized embeddings")
            vectors.append(value); cursor += len(batch)
        except RuntimeError as exc:
            if not is_oom(exc) or size <= MIN_BATCH_SIZE: raise
            size = max(MIN_BATCH_SIZE, size // 2)
            torch.cuda.empty_cache()  # only after an actual CUDA OOM retry
    return np.ascontiguousarray(np.concatenate(vectors)), size


In [ ]:
# Artifact creation. It uses a temporary run directory and publishes only after complete validation.
def write_sql(path: Path, metadata: pd.DataFrame) -> None:
    lines = ["-- frame.faiss and this SQL are an inseparable full-rebuild pair.", "-- Import only into a synchronized replaced/new index; never append to production."]
    for row in metadata.itertuples(index=False):
        frame_id = str(row.frame_id).replace("'", "''")
        lines.append(f"INSERT INTO frameembeddingrecord (faiss_id, index_version, frame_id, model_name) VALUES ({row.faiss_id}, {INDEX_VERSION}, '{frame_id}', '{MODEL_NAME}') ON CONFLICT (frame_id, index_version) DO NOTHING;")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")

def validate_run(run_dir: Path, dimension: int, expected_success: int) -> dict[str, str]:
    metadata_paths = sorted((run_dir / "metadata").glob("part-*.parquet")); vector_paths = sorted((run_dir / "vectors").glob("part-*.npy"))
    metadata = pd.concat([pd.read_parquet(path) for path in metadata_paths], ignore_index=True) if metadata_paths else pd.DataFrame()
    rows = 0
    for path in vector_paths:
        value = np.load(path, mmap_mode="r")
        assert value.dtype == np.float32 and value.ndim == 2 and value.shape[1] == dimension and np.isfinite(value).all()
        assert np.allclose(np.linalg.norm(value, axis=1), 1.0, atol=1e-4); rows += len(value)
    index = faiss.read_index(str(run_dir / "frame.faiss"))
    assert rows == len(metadata) == expected_success == index.ntotal
    assert metadata.faiss_id.is_unique and metadata.frame_id.is_unique
    checksums = {str(path.relative_to(run_dir)): sha256_file(path) for path in run_dir.rglob("*") if path.is_file() and path.name != "manifest.json"}
    return checksums

def run_embedding() -> Path:
    run_id = RUN_ID or f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_v{INDEX_VERSION}_{MODEL_NAME}"
    final_dir, temp_dir = OUTPUT_DIR / run_id, OUTPUT_DIR / (run_id + ".tmp-" + uuid.uuid4().hex)
    if final_dir.exists(): raise FileExistsError(f"Refusing to overwrite existing run: {final_dir}")
    temp_dir.mkdir(parents=True); (temp_dir / "vectors").mkdir(); (temp_dir / "metadata").mkdir()
    failures = temp_dir / "failures.jsonl"; failures.touch()
    checkpoint = temp_dir / "checkpoint.json"
    state = {"next_faiss_id": 1, "state": "in_progress"}
    atomic_json(checkpoint, state)
    rows, failure_count = load_rows(failures); started = time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    loader_args = dict(batch_size=IMAGE_BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate)
    if NUM_WORKERS: loader_args.update(persistent_workers=True, prefetch_factor=2)
    loader = DataLoader(FrameDataset(rows), **loader_args)
    index, dimension, success, pending_vectors, pending_rows, final_batch = None, None, 0, [], [], IMAGE_BATCH_SIZE
    shard_number = 0
    def flush() -> None:
        nonlocal shard_number, pending_vectors, pending_rows
        if not pending_vectors: return
        array = np.ascontiguousarray(np.concatenate(pending_vectors), dtype=np.float32)
        np.save(temp_dir / "vectors" / f"part-{shard_number:05d}.npy", array)
        pd.DataFrame(pending_rows).to_parquet(temp_dir / "metadata" / f"part-{shard_number:05d}.parquet", index=False)
        pending_vectors, pending_rows = [], []; shard_number += 1
        atomic_json(checkpoint, {**state, "last_committed_shard": shard_number - 1})
        atomic_json(checkpoint, {**state, "last_committed_shard": shard_number - 1})
    for samples in loader:
        good = [(i, image) for i, image, reason in samples if image is not None]
        for i, image, reason in samples:
            if reason: append_failure(failures, rows[i]["frame_id"], reason, rows[i]["frame_path"]); failure_count += 1
        if not good: continue
        vectors, final_batch = encode_images([image for _, image in good], final_batch)
        if dimension is None: dimension = int(vectors.shape[1]); index = faiss.IndexIDMap2(faiss.IndexFlatIP(dimension))
        if vectors.shape[1] != dimension: raise RuntimeError("embedding dimension changed during run")
        ids = np.arange(state["next_faiss_id"], state["next_faiss_id"] + len(good), dtype=np.int64); index.add_with_ids(vectors, ids)
        state["next_faiss_id"] += len(good)
        for (i, _), vector, faiss_id in zip(good, vectors, ids):
            row = rows[i]; pending_vectors.append(vector[None, :]); pending_rows.append({"faiss_id": int(faiss_id), "frame_id": row["frame_id"], "video_id": row["video_id"], "shot_id": row["shot_id"], "timestamp_ms": row["timestamp_ms"], "source": row["source"], "frame_path": row["frame_path"], "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "dimension": dimension, "normalized": True, "vector_shard": shard_number, "vector_row": len(pending_rows) - 1})
            success += 1
            if len(pending_rows) >= ROWS_PER_SHARD: flush()
    if dimension is None: raise RuntimeError("No valid images were embedded; no FAISS/SQL pair is produced")
    flush(); faiss.write_index(index, str(temp_dir / "frame.faiss"))
    metadata = pd.concat([pd.read_parquet(path) for path in sorted((temp_dir / "metadata").glob("*.parquet"))], ignore_index=True)
    metadata[["faiss_id"]].assign(index_version=INDEX_VERSION, model_name=MODEL_NAME, frame_id=metadata.frame_id)[["faiss_id", "index_version", "frame_id", "model_name"]].to_csv(temp_dir / "frame_embedding_mapping.csv", index=False)
    write_sql(temp_dir / "insert_frame_embedding_records.sql", metadata)
    checksums = validate_run(temp_dir, dimension, success)
    elapsed = time.perf_counter() - started
    summary = {"gpu_name": GPU_NAME, "cuda_version": torch.version.cuda, "torch_version": torch.__version__, "compute_dtype": str(COMPUTE_DTYPE), "initial_batch_size": IMAGE_BATCH_SIZE, "final_batch_size": final_batch, "images_per_second": success / elapsed if elapsed else 0, "peak_vram_bytes": torch.cuda.max_memory_allocated(), "input_count": len(rows), "success_count": success, "failure_count": failure_count, "dimension": dimension}
    atomic_json(temp_dir / "summary.json", summary)
    manifest = {"model": {"id": MODEL_ID, "revision": MODEL_REVISION, "name": MODEL_NAME, "backend": MODEL_BACKEND}, "config": {"index_version": INDEX_VERSION, "video_start": VIDEO_START, "video_end": VIDEO_END, "rows_per_shard": ROWS_PER_SHARD}, "input_sha256": sha256_file(KEYFRAME_FILE), "dimension": dimension, "success_count": success, "failure_count": failure_count, "artifacts_sha256": checksums, "frame_faiss_sha256": sha256_file(temp_dir / "frame.faiss")}
    atomic_json(temp_dir / "manifest.json", manifest)
    temp_dir.replace(final_dir); return final_dir


In [ ]:
# Mandatory GPU smoke and negative tests. It loads the fixed model and never touches the real dataset.
def required_tests() -> None:
    colors = [Image.new("RGB", (32, 32), color) for color in ("red", "blue")]
    vectors, _ = encode_images(colors, min(2, IMAGE_BATCH_SIZE))
    assert vectors.shape[0] == 2 and vectors.dtype == np.float32 and np.isfinite(vectors).all()
    assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-4)
    smoke = faiss.IndexIDMap2(faiss.IndexFlatIP(vectors.shape[1])); ids = np.array([1, 2], dtype=np.int64); smoke.add_with_ids(vectors, ids)
    _, found = smoke.search(vectors, 1); assert len(set(ids)) == 2 and np.array_equal(found[:, 0], ids)
    test_failures = Path("/kaggle/working/frame_embedding_negative_failures.jsonl"); test_failures.unlink(missing_ok=True)
    append_failure(test_failures, "missing-frame", "image_not_found", "does-not-exist.png")
    assert len(test_failures.read_text(encoding="utf-8").splitlines()) == 1
    print("Mandatory smoke and negative tests passed; dimension=", vectors.shape[1])

required_tests()


In [ ]:
# URL image cache: run after input validation and before the real run.
# The preflight requires Content-Length, so the 15 GiB Kaggle working limit is enforced before workers write files.
from concurrent.futures import ThreadPoolExecutor, as_completed
import shutil, urllib.parse, urllib.request
DOWNLOAD_WORKERS = 8
MAX_WORKING_BYTES = 15 * 1024 ** 3
IMAGE_CACHE_DIR = OUTPUT_DIR / "image_cache"

def _working_bytes() -> int:
    return sum(path.stat().st_size for path in OUTPUT_DIR.rglob("*") if path.is_file()) if OUTPUT_DIR.exists() else 0

def _download_image(row: dict[str, Any]) -> Path:
    url = str(row.get("image_url", "")).strip()
    if not url.startswith(("https://", "http://")):
        raise RuntimeError(f"{row['frame_id']}: image_url is required when no local image exists")
    suffix = Path(urllib.parse.urlsplit(url).path).suffix.lower() or ".jpg"
    target = IMAGE_CACHE_DIR / f"{row['frame_id']}{suffix}"
    if target.is_file() and target.stat().st_size:
        return target
    part = target.with_suffix(target.suffix + ".part")
    try:
        with urllib.request.urlopen(url, timeout=120) as response, part.open("wb") as handle:
            shutil.copyfileobj(response, handle, length=1024 * 1024)
        if part.stat().st_size == 0:
            raise RuntimeError(f"{row['frame_id']}: empty image download")
        part.replace(target)
        return target
    finally:
        part.unlink(missing_ok=True)

def download_missing_images(rows: list[dict[str, Any]]) -> None:
    pending = [row for row in rows if row["image_path"] is None]
    if not pending:
        return
    IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    sizes = []
    for row in pending:
        url = str(row.get("image_url", "")).strip()
        request = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(request, timeout=30) as response:
            value = response.headers.get("Content-Length")
        if value is None:
            raise RuntimeError(f"{row['frame_id']}: URL must provide Content-Length for the 15 GiB guard")
        sizes.append(int(value))
    if _working_bytes() + sum(sizes) > MAX_WORKING_BYTES:
        raise RuntimeError("Refusing downloads: Kaggle working output would exceed 15 GiB")
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = {pool.submit(_download_image, row): row for row in pending}
        for future in as_completed(futures):
            futures[future]["image_path"] = future.result()

_original_load_rows = load_rows
def load_rows(failures: Path) -> tuple[list[dict[str, Any]], int]:
    rows, failure_count = _original_load_rows(failures)
    download_missing_images(rows)
    return rows, failure_count


In [ ]:
# Real run. Run this cell after reviewing the configuration. Download the resulting folder from Kaggle Output.
run_dir = run_embedding()
print(f"Completed: {run_dir}")
print("Kaggle: save the notebook version, then download the folder from the Output pane. Do not run the generated SQL automatically.")

# Downloaded images are no longer needed after a successful published embedding run.
if IMAGE_CACHE_DIR.exists():
    shutil.rmtree(IMAGE_CACHE_DIR)
    print("Removed downloaded image cache after successful run.")
